# In Scikit-Learn nutzt man dafür den CountVectorizer und TfidfVectorizer
## Achtung: CountVectorizer und TfidfVectorizer sind eigentlich nicht so geeignet für Klassifikation. TfidfVectorizer ist aber besser als CountVectorizer. Das Problem ist dass die Anzahl der Spalten wächst mit der Anzahl der Wörter bei CountVectorizer und dadurch dass die Anzahl der Spalten nicht viel kleiner als die Anzahl der Zeilen ist. Das ist instabil beo LogisticProgressor.

Das ist das Standard-Werkzeug, um Text in eine numerische Matrix (Bag-of-Words) umzuwandeln.
Interne Verarbeitungsschritte

Der CountVectorizer führt automatisch mehrere NLP-Schritte aus:

Lowercasing

    "NLP ist gut" → "nlp ist gut"

Tokenisierung

    "nlp ist gut" → ["nlp","ist","gut"]

Vokabular aufbauen

    {"nlp":0, "ist":1, "gut":2}

Wortzählung

    [1,1,1]

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Beispieldaten (Liste von Sätzen)
corpus = [
    "I love NLP.",
    "NLP is amazing and NLP is fun.",
    "I love fun things."
]

# 2. Vectorizer initialisieren
vectorizer = CountVectorizer()

# 3. Den Text in die Matrix umwandeln (Lernen + Transformieren)
X = vectorizer.fit_transform(corpus)
print("X:", X)
# --- Ergebnisse anschauen ---

# Alle gefundenen Wörter (das Vokabular)
print("Vokabular (Wörter):", vectorizer.get_feature_names_out())

# Die IDs der Wörter
print("Wort-IDs:", vectorizer.vocabulary_)

# Die fertige Matrix (als Array dargestellt)
print("Bag-of-Words Matrix:\n", X.toarray())



X: <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 10 stored elements and shape (3, 7)>
  Coords	Values
  (0, 4)	1
  (0, 5)	1
  (1, 5)	2
  (1, 3)	2
  (1, 0)	1
  (1, 1)	1
  (1, 2)	1
  (2, 4)	1
  (2, 2)	1
  (2, 6)	1
Vokabular (Wörter): ['amazing' 'and' 'fun' 'is' 'love' 'nlp' 'things']
Wort-IDs: {'love': 4, 'nlp': 5, 'is': 3, 'amazing': 0, 'and': 1, 'fun': 2, 'things': 6}
Bag-of-Words Matrix:
 [[0 0 0 0 1 1 0]
 [1 1 1 2 0 2 0]
 [0 0 1 0 1 0 1]]


In [21]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, accuracy_score

# -------------------------------------------------
# 1. Datensatz (50 Texte)
# -------------------------------------------------

texts = [
"I love this product",
"This is amazing",
"Very good quality",
"Excellent item",
"I am very happy",
"This works perfectly",
"Absolutely fantastic",
"Great purchase",
"Really satisfied",
"I recommend this",
"Super product",
"Very nice experience",
"I like it a lot",
"Wonderful quality",
"Best thing I bought",
"Very reliable product",
"Five stars",
"Totally worth it",
"Very impressive",
"Outstanding item",
"Good value for money",
"Extremely satisfied",
"I enjoy using this",
"Perfect choice",
"High quality product",

"I hate this product",
"This is terrible",
"Very bad quality",
"Worst item ever",
"I am disappointed",
"This does not work",
"Absolutely awful",
"Bad purchase",
"Really dissatisfied",
"I do not recommend this",
"Horrible product",
"Very poor experience",
"I dislike it",
"Terrible quality",
"Waste of money",
"Very unreliable product",
"One star",
"Totally useless",
"Very disappointing",
"Awful item",
"Bad value for money",
"Extremely dissatisfied",
"I regret buying this",
"Very poor choice",
"Low quality product"
]

labels = [
"pos","pos","pos","pos","pos","pos","pos","pos","pos","pos",
"pos","pos","pos","pos","pos","pos","pos","pos","pos","pos",
"pos","pos","pos","pos","pos",

"neg","neg","neg","neg","neg","neg","neg","neg","neg","neg",
"neg","neg","neg","neg","neg","neg","neg","neg","neg","neg",
"neg","neg","neg","neg","neg"
]

# -------------------------------------------------
# 2. Train/Test Split
# -------------------------------------------------


train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    shuffle=True,
    stratify=labels,
    random_state=42
)


# -------------------------------------------------
# 3. Pipeline
# -------------------------------------------------
"""
pipeline = Pipeline([
    ("vectorizer", CountVectorizer(max_features=30)),
    ("model", LogisticRegression())
])
"""

from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("vectorizer", CountVectorizer(max_features=25)),
    ("model", RandomForestClassifier(n_estimators=50, random_state=42))
])



# -------------------------------------------------
# 4. Training
# -------------------------------------------------

pipeline.fit(train_texts, train_labels)

vectorizer = pipeline.named_steps["vectorizer"]

# Anzahl Features / Spalten
num_features = len(vectorizer.get_feature_names_out())
print("Anzahl Features:", num_features)

# Optional: ein paar Feature-Namen ausgeben
print("Beispiel Features:", vectorizer.get_feature_names_out()[:20])


# -------------------------------------------------
# 5. Prediction
# -------------------------------------------------

predictions = pipeline.predict(test_texts)

# -------------------------------------------------
# 6. Evaluation
# -------------------------------------------------

print("Test Texte:")
print(test_texts)
print("test_labels:")
print(test_labels)

print("\nVorhersagen:")
print(predictions)

print("\nAccuracy:")
print(accuracy_score(test_labels, predictions))

print("\nClassification Report:")
print(classification_report(test_labels, predictions))

print("Train size:", len(train_texts))
print("Vocabulary size:", len(pipeline.named_steps["vectorizer"].vocabulary_))

print(pipeline.predict(["wonderful quality"]))
print(pipeline.predict(["terrible product"]))

Anzahl Features: 25
Beispiel Features: ['am' 'bad' 'choice' 'dislike' 'dissatisfied' 'do' 'does' 'enjoy' 'ever'
 'for' 'good' 'is' 'it' 'item' 'money' 'not' 'poor' 'product' 'quality'
 'really']
Test Texte:
['Horrible product', 'Absolutely awful', 'Waste of money', 'Awful item', 'I recommend this', 'High quality product', 'Five stars', 'Very nice experience', 'Bad purchase', 'Extremely satisfied']
test_labels:
['neg', 'neg', 'neg', 'neg', 'pos', 'pos', 'pos', 'pos', 'neg', 'pos']

Vorhersagen:
['pos' 'pos' 'pos' 'pos' 'pos' 'neg' 'pos' 'pos' 'neg' 'pos']

Accuracy:
0.5

Classification Report:
              precision    recall  f1-score   support

         neg       0.50      0.20      0.29         5
         pos       0.50      0.80      0.62         5

    accuracy                           0.50        10
   macro avg       0.50      0.50      0.45        10
weighted avg       0.50      0.50      0.45        10

Train size: 40
Vocabulary size: 25
['pos']
['pos']


#  TfidfVectorizer: Tokenizer & Modell laden
text cleaning:

Orig. Text-> stopword-> text.lower()-> lemmatize-> string.punctuation

In [23]:
texts = [
    "I love this movie. It is amazing!",
    "This film was terrible and boring.",
    "Absolutely fantastic experience.",
    "I hate this product. Very bad quality.",
    "Best purchase ever!",
    "Worst thing I have bought."
]

labels = [1, 0, 1, 0, 1, 0]  # 1 = positiv, 0 = negativ

import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#nltk.download('punkt')
#nltk.download('stopwords')
#nltk.download('wordnet')
print("staz zeichen:\n", string.punctuation)
print()
stop_words = set(stopwords.words("english"))
print("stop_words:\n", stop_words)
print()
# WICHTIG: Negationen behalten!
stop_words.discard("not")

lemmatizer = WordNetLemmatizer()

print("remove stop words, text.lower() and lemmatize :")
def preprocess(text):
    tokens = word_tokenize(text.lower())
    print("tokens 1:", tokens)
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
        and word not in string.punctuation
    ]

    print("tokens 2:", tokens)
    return " ".join(tokens)

processed_texts = [preprocess(t) for t in texts]
print()
print("processed_texts:",processed_texts)


staz zeichen:
 !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

stop_words:
 {'s', 'whom', 'can', "you've", 'no', 'against', 'does', 'we', 'again', "i'm", 'than', 'our', 'wasn', "that'll", 'ain', 'more', 'doesn', 'where', "shouldn't", 'under', 'm', 'd', 'what', 'same', 'of', 'own', 'their', "you'll", 'all', 'now', 'here', 'those', 'needn', 'not', 'such', 'at', 'but', "hasn't", 're', 'having', 'itself', 'o', "we've", 'with', 'doing', 'the', 'hasn', 'shan', 'myself', 'into', 'isn', "we'd", 'ourselves', 'below', 'ours', "they're", 'shouldn', "i'll", 'for', "they've", 'until', 'any', "wouldn't", 'y', 'that', 'he', "mightn't", 'there', "doesn't", 'above', 'a', 'has', 'to', 'your', 'some', 'do', 'just', 'was', 'once', 'i', 'an', 'during', 'mustn', "i'd", "it'll", "she'll", "she's", "mustn't", 'by', 'each', 'll', 'aren', 't', 'out', 'yourself', 'hadn', "it's", 'his', 'are', 'hers', 'too', 'after', 'it', 'very', 'as', 'or', 'didn', 'in', "she'd", 'herself', 'will', 'couldn', 'other', 'her', 've', "should've"

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

#vectorizer = TfidfVectorizer(ngram_range=(1,2),  max_features=500)  # Uni + Bigram
vectorizer = TfidfVectorizer(ngram_range=(1,2))  # Uni + Bigram
X = vectorizer.fit_transform(processed_texts)

print("Feature-Anzahl:", len(vectorizer.get_feature_names_out()))
print("TF-IDF Matrix Shape:", X.shape)
print(vectorizer.get_feature_names_out())
print("X:", X)

Feature-Anzahl: 32
TF-IDF Matrix Shape: (6, 32)
['absolutely' 'absolutely fantastic' 'amazing' 'bad' 'bad quality' 'best'
 'best purchase' 'boring' 'bought' 'ever' 'experience' 'fantastic'
 'fantastic experience' 'film' 'film terrible' 'hate' 'hate product'
 'love' 'love movie' 'movie' 'movie amazing' 'product' 'product bad'
 'purchase' 'purchase ever' 'quality' 'terrible' 'terrible boring' 'thing'
 'thing bought' 'worst' 'worst thing']
X: <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 32 stored elements and shape (6, 32)>
  Coords	Values
  (0, 17)	0.4472135954999579
  (0, 19)	0.4472135954999579
  (0, 2)	0.4472135954999579
  (0, 18)	0.4472135954999579
  (0, 20)	0.4472135954999579
  (1, 13)	0.4472135954999579
  (1, 26)	0.4472135954999579
  (1, 7)	0.4472135954999579
  (1, 14)	0.4472135954999579
  (1, 27)	0.4472135954999579
  (2, 0)	0.4472135954999579
  (2, 11)	0.4472135954999579
  (2, 10)	0.4472135954999579
  (2, 1)	0.4472135954999579
  (2, 12)	0.4472135954999579
  (3, 15)

In [36]:
import torch
import torch.nn as nn
import torch.optim as optim

X_tensor = torch.tensor(X.toarray(), dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
print("X_tensor:\n", X_tensor)
print("y_tensor:\n", y_tensor)

X_tensor:
 tensor([[0.0000, 0.0000, 0.4472, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4472,
         0.4472, 0.4472, 0.4472, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4472, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.4472, 0.4472, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4472,
         0.4472, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4472, 0.4472, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.4472, 0.4472, 0.4472, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3780, 0.3780, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.00

# Model: LogisticRegression

In [37]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        return torch.sigmoid(self.linear(x))
print("X_tensor.shape:", X_tensor.shape)
model = LogisticRegression(X_tensor.shape[1])

X_tensor.shape: torch.Size([6, 32])


# training

In [38]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(201):
    optimizer.zero_grad()
    
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.6895
Epoch 50, Loss: 0.2963
Epoch 100, Loss: 0.1516
Epoch 150, Loss: 0.0927
Epoch 200, Loss: 0.0634


# Evaluation

In [39]:
with torch.no_grad():
    predictions = model(X_tensor)
    predicted_classes = (predictions > 0.5).float()
    print("predicted_classes:", predicted_classes)
    
    
accuracy = (predicted_classes == y_tensor).float().mean()

print("Accuracy:", accuracy.item())

predicted_classes: tensor([[1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.]])
Accuracy: 1.0


# Neue Vorhersage testen

In [32]:
new_text = "This was not good at all"

new_processed = preprocess(new_text)
new_vector = vectorizer.transform([new_processed])
new_tensor = torch.tensor(new_vector.toarray(), dtype=torch.float32)

with torch.no_grad():
    prediction = model(new_tensor)
    
print("Sentiment Score:", prediction.item())

tokens 1: ['this', 'was', 'not', 'good', 'at', 'all']
tokens 2: ['not', 'good']
Sentiment Score: 0.5307232141494751


# LogisticRegression

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Beispieldaten: Rezensionen und Labels (1 = positiv, 0 = negativ)
reviews = [
    "This movie was amazing and great", 
    "I hated this film, it was boring",
    "Excellent storytelling and acting",
    "Waste of time, terrible experience",
    "Best movie of the year",
    "I fell asleep, so bad"
]
labels = [1, 0, 1, 0, 1, 0] # 1: Positiv, 0: Negativ

# 2. Text in Zahlen umwandeln (TF-IDF)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(reviews)

# 3. Das Modell trainieren
model = LogisticRegression()
model.fit(X, labels)

# 4. Vorhersage für neue, unbekannte Sätze
new_reviews = ["What a great acting", "It was a boring waste of time"]
X_new = vectorizer.transform(new_reviews) # WICHTIG: Nur transform, nicht fit!

predictions = model.predict(X_new)

# 5. Ergebnis ausgeben
for review, pred in zip(new_reviews, predictions):
    sentiment = "Positiv" if pred == 1 else "Negativ"
    print(f"Rezension: '{review}' -> Vorhergesagtes Sentiment: {sentiment}")


Rezension: 'What a great acting' -> Vorhergesagtes Sentiment: Positiv
Rezension: 'It was a boring waste of time' -> Vorhergesagtes Sentiment: Negativ


#  Random Forest

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 1. Beispieldaten
reviews = [
    "This movie was amazing and great", 
    "I hated this film, it was boring",
    "Excellent storytelling and acting",
    "Waste of time, terrible experience",
    "Best movie of the year",
    "I fell asleep, so bad"
]
labels = [1, 0, 1, 0, 1, 0] # 1: Positiv, 0: Negativ

# 2. Text in TF-IDF Matrix umwandeln
# Wir nutzen Bigramme, damit "not good" besser erkannt wird
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(reviews)

# 3. Random Forest Modell erstellen und trainieren
# n_estimators=100 bedeutet, wir nutzen 100 kleine Entscheidungsbäume
#rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model = LogisticRegression()
rf_model.fit(X, labels)

# 4. Vorhersage für neue Sätze
test_sentences = ["An amazing experience", "It was so boring and bad"]
X_test = vectorizer.transform(test_sentences)

predictions = rf_model.predict(X_test)
# Wahrscheinlichkeiten anzeigen (wie sicher ist sich das Modell?)
probabilities = rf_model.predict_proba(X_test)

# 5. Ergebnis-Ausgabe
for i, sentence in enumerate(test_sentences):
    sentiment = "Positiv" if predictions[i] == 1 else "Negativ"
    confidence = max(probabilities[i]) * 100
    print(f"Satz: '{sentence}' -> {sentiment} ({confidence:.1f}% sicher)")


Satz: 'An amazing experience' -> Negativ (50.7% sicher)
Satz: 'It was so boring and bad' -> Negativ (55.3% sicher)


# RandomForestClassifier or LogisticRegression 
## TfidfVectorizer
with more data

TfidfVectorizer(stop_words='english'): default: none

    max_df (Maximum Document Frequency): TfidfVectorizer(max_df=0.7)
    min_df (Minimum Document Frequency): TfidfVectorizer(min_df=2

## Der ideale Workflow für dein Projekt:

    Bereinigung: HTML-Tags, Sonderzeichen und Zahlen entfernen.
    Tokenisierung & Lemmatisierung: Den Text in Grundformen zerlegen (z. B. mit spaCy).
    Stopword-Filter: Die bereinigten Grundformen filtern.
    Vectorizing: Den sauberen Text in den TfidfVectorizer füttern.
    Klassifikation: Das Ergebnis in ein Modell (z. B. Logistic Regression oder SVM) geben.
    
    Tipp: Entferne „nicht“ oder „kein“ aus deiner Stopword-Liste, bevor du sie an den Vectorizer übergibst.
    

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 1. Erweiterte Trainingsdaten (50 Sätze)
reviews = [
    # POSITIV (25)
    "I love this movie", "Amazing experience", "Great acting and plot", 
    "Best film of the year", "Fantastic storytelling", "I really enjoyed it",
    "A masterpiece of cinema", "Brilliant performance", "Highly recommended",
    "So much fun to watch", "Excellent direction", "A wonderful journey",
    "Pure joy from start to finish", "Stunning visuals", "Very moving story",
    "I would watch it again", "Top notch acting", "Incredible soundtrack",
    "The best I have seen", "An emotional rollercoaster in a good way",
    "Simply beautiful", "A classic for sure", "Powerfully acted", 
    "Surprisingly good", "A delight for the eyes",
    
    # NEGATIV (25)
    "I hated this movie", "Terrible experience", "Boring plot and bad acting",
    "Worst film ever", "Waste of time", "I did not like it",
    "A total disaster", "Poor performance", "Do not recommend",
    "A boring mess", "Horrible direction", "A painful journey",
    "Dull from start to finish", "Ugly visuals", "Very weak story",
    "I would never watch it again", "Low quality production", "Annoying soundtrack",
    "The worst I have seen", "An emotional mess",
    "Simply ugly", "A failure for sure", "Weakly acted", 
    "Disappointingly bad", "A pain to watch"
]

# Labels: 25 mal 1 (Positiv), 25 mal 0 (Negativ)
labels = [1]*25 + [0]*25

# 2. Modellierung
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
print("X-shape:", X.shape)
X = vectorizer.fit_transform(reviews)
model = RandomForestClassifier(n_estimators=100, random_state=42)
#model = LogisticRegression()
model.fit(X, labels)

# 3. Test mit den kritischen Sätzen
test_sentences = ["An amazing experience", "It was so boring and bad"]
X_test = vectorizer.transform(test_sentences)

predictions = model.predict(X_test)
probs = model.predict_proba(X_test)

# 4. Ausgabe
# Korrigierte Ausgabe (ohne [INDEX])
for i, sentence in enumerate(test_sentences):
    sentiment = "Positiv" if predictions[i] == 1 else "Negativ"
    conf = max(probs[i]) * 100
    print(f"Satz: '{sentence}' -> {sentiment} ({conf:.1f}% sicher)")



X-shape: (50, 189)
Satz: 'An amazing experience' -> Positiv (64.0% sicher)
Satz: 'It was so boring and bad' -> Negativ (69.0% sicher)
